# So sánh mô hình: dataset 1h và 1h_top4

Chỉ đọc hai experiment **1h** và **1h_top4**. Chọn run FINISHED có `test_loss` thấp nhất **riêng cho từng experiment + model + feature set**. Các bảng luôn giữ tên experiment, không gộp hai dataset để chọn một run duy nhất.

**Đơn vị: 1 đơn vị RMSE = 1 mm; MAE và Peak Error cũng theo mm; MSE theo mm².** Ví dụ RMSE 21,073 mm tương đương 2,107 cm. NSE/KGE không có đơn vị; sai lệch thời gian đỉnh theo phút. Quy ước mm dựa trên việc đối chiếu dữ liệu Dã Viên với các bản tin mực nước; API chưa công bố đơn vị trong schema.

Metric mùa lũ và ngoài mùa lũ được tính từ dữ liệu dự đoán, tuyệt đối không train lại. Mùa lũ là tháng 9–12 theo giờ Việt Nam, phân nhóm theo **thời điểm phát dự báo t**; mỗi mục tiêu vẫn là mực nước tại t + horizon. Peak Error là độ lệch giữa hai giá trị đỉnh cao nhất trong nhóm, không phải sai số tại mọi đỉnh của từng đợt lũ.

Kết quả toàn tập Test được lấy từ MLflow. Hai dataset và độ dài chuỗi có thể dùng các mẫu Test khác nhau; bảng so sánh mô tả kết quả trên Test của từng run, không chứng minh riêng việc đổi feature đã gây cải thiện. `test_loss` chỉ dùng chọn run trong cùng nhóm, không dùng so sánh cây với DL. Trung bình các horizon chỉ để xem tổng quát.

In [1]:
import ast
import importlib.util
import os
import re
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient

# None: dùng URI đã set trong environment/session hoặc cấu hình src/ai/train.py.
TRACKING_URI = None
# Chỉ đọc hai experiment cần so sánh; không lấy experiment 10min.
EXPERIMENT_NAMES = ["1h", "1h_top4"]
# Bao gồm cả run đã lưu trữ/xóa mềm; đổi thành ACTIVE_ONLY nếu chỉ cần run hiện hành.
RUN_VIEW_TYPE = ViewType.ALL
HORIZONS = [60, 180, 360, 720, 1440]
METRICS = ["rmse", "mae", "nse", "kge", "peak_absolute_error", "peak_timing_error_minutes"]
MODEL_TYPES = ["XGBoost", "LightGBM", "Vanilla LSTM", "Residual LSTM", "Adapter LSTM"]
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src/ai/train.py").is_file()), Path.cwd())
NOTEBOOK_PATH = PROJECT_ROOT / "scripts/mlflow_metrics.ipynb"
OUTPUT_DIR = PROJECT_ROOT / "reports/mlflow_metrics"


def configured_tracking_uri():
    """Read the existing project constant without importing/running train.py."""
    source = PROJECT_ROOT / "src/ai/train.py"
    if source.is_file():
        for node in ast.parse(source.read_text(encoding="utf-8")).body:
            if isinstance(node, ast.Assign) and any(
                isinstance(target, ast.Name) and target.id == "MLFLOW_TRACKING_URI"
                for target in node.targets
            ) and isinstance(node.value, ast.Constant):
                return node.value.value
    return None


current_uri = mlflow.get_tracking_uri()
# Check explicit session configuration, avoiding the default local SQLite store
# when the project already points to an HTTP tracking server.
try:
    from mlflow.tracking._tracking_service.utils import is_tracking_uri_set
    session_uri = current_uri if is_tracking_uri_set() else None
except ImportError:
    session_uri = current_uri
tracking_uri = TRACKING_URI or os.environ.get("MLFLOW_TRACKING_URI") or session_uri or configured_tracking_uri() or current_uri
print("mlflow.get_tracking_uri():", current_uri)
print("Tracking URI được dùng:", tracking_uri)
print("Thư mục export:", OUTPUT_DIR)
client = MlflowClient(tracking_uri=tracking_uri)

mlflow.get_tracking_uri(): sqlite:////home/my_ubuntu/projects/hydrology/mlflow.db
Tracking URI được dùng: http://localhost:5000
Thư mục export: /home/my_ubuntu/projects/hydrology/scripts


## 1. Đọc toàn bộ experiment và run FINISHED

Các API search có phân trang. Vòng lặp dưới đây đọc hết các trang, không dừng ở giới hạn mặc định. Nếu API có lỗi, notebook báo lỗi thay vì xuất một báo cáo thiếu dữ liệu mà không thông báo.


In [2]:
def read_all_experiments(client):
    experiments, token = [], None
    while True:
        page = client.search_experiments(view_type=ViewType.ALL, max_results=100, page_token=token)
        experiments.extend(page)
        token = page.token
        if not token:
            return experiments


def read_finished_runs(client, experiment_id):
    runs, token = [], None
    while True:
        page = client.search_runs(
            experiment_ids=[experiment_id], filter_string="attributes.status = 'FINISHED'",
            run_view_type=RUN_VIEW_TYPE, max_results=1000,
            order_by=["attributes.start_time DESC"], page_token=token,
        )
        runs.extend(run for run in page if run.info.status == "FINISHED")
        token = page.token
        if not token:
            return runs


all_experiments = read_all_experiments(client)
experiments = all_experiments
if EXPERIMENT_NAMES is not None:
    wanted = set(EXPERIMENT_NAMES)
    missing = wanted - {experiment.name for experiment in all_experiments}
    if missing:
        raise ValueError(f"Không tìm thấy experiment: {sorted(missing)}")
    experiments = [experiment for experiment in all_experiments if experiment.name in wanted]
finished_runs = []
experiment_by_id = {experiment.experiment_id: experiment.name for experiment in experiments}
for experiment in experiments:
    finished_runs.extend(read_finished_runs(client, experiment.experiment_id))
runs_by_id = {run.info.run_id: run for run in finished_runs}
print(f"Đã tìm thấy {len(all_experiments)} experiment; đọc run từ {len(experiments)} experiment.")
print(f"Số run FINISHED: {len(finished_runs)}")

# Kiểm tra metadata thực tế trước khi chuẩn hóa: một ví dụ cho mỗi train_method.
preview, seen = [], set()
for run in finished_runs:
    method = run.data.params.get("train_method", "(không có)")
    if method in seen:
        continue
    seen.add(method)
    preview.append({
        "experiment_name": experiment_by_id[run.info.experiment_id],
        "run_name": run.data.tags.get("mlflow.runName", getattr(run.info, "run_name", "")),
        "params": {key: run.data.params[key] for key in
                   ["train_method", "model_type", "feature_set", "horizon", "dataset_dir"]
                   if key in run.data.params},
        "tags": {key: run.data.tags[key] for key in
                 ["model_type", "feature_set", "test_loss_scale", "best_checkpoint_stage"]
                 if key in run.data.tags},
    })
display(pd.DataFrame(preview))


Đã tìm thấy 5 experiment; đọc run từ 2 experiment.
Số run FINISHED: 116


,experiment_name,run_name,params,tags
0,1h_top4,lgbm_1h_water_level+rain+flood_season,"{'train_method': 'lgbm', 'feature_set': 'water_level+rain+flood_season', 'horizon': '1h', 'dataset_dir': '/home/my_ubuntu/projects/hydrology/train_data/1h_top4'}",{'test_loss_scale': 'original_target_units'}
1,1h_top4,xgb_1h_water_level+rain+flood_season,"{'train_method': 'xgb', 'feature_set': 'water_level+rain+flood_season', 'horizon': '1h', 'dataset_dir': '/home/my_ubuntu/projects/hydrology/train_data/1h_top4'}",{'test_loss_scale': 'original_target_units'}
2,1h_top4,lstm_1h_water_level+rain+flood_season,"{'train_method': 'lstm', 'feature_set': 'water_level+rain+flood_season', 'horizon': '1h', 'dataset_dir': '/home/my_ubuntu/projects/hydrology/train_data/1h_top4'}",{'test_loss_scale': 'scaled_target_units'}
3,1h_top4,lstm_adapter_1h_water_level+rain+flood_season_two_stage,"{'train_method': 'lstm_adapter', 'feature_set': 'water_level+rain+flood_season', 'horizon': '1h', 'dataset_dir': '/home/my_ubuntu/projects/hydrology/train_data/1h_top4'}","{'test_loss_scale': 'scaled_target_units', 'best_checkpoint_stage': '1'}"
4,1h_top4,residual_lstm_1h_water_level+flood_season,"{'train_method': 'residual_lstm', 'feature_set': 'water_level+flood_season', 'horizon': '1h', 'dataset_dir': '/home/my_ubuntu/projects/hydrology/train_data/1h_top4'}",{'test_loss_scale': 'scaled_target_units'}


## 2. Nhận diện model và feature set

Ưu tiên các params/tags đã lưu. Nếu thiếu, parser nhận diện tên model ở đầu tên run, bỏ phần độ phân giải thời gian và các hậu tố backbone đã biết. Những run không nhận diện được feature set sẽ được liệt kê để kiểm tra, thay vì gộp vào một nhóm đoán sai. `water` và `water_level` được xem là cùng nhóm; thứ tự các nhóm không làm tạo thêm nhóm kết quả.


In [3]:
MODEL_ALIASES = {
    "xgb": "XGBoost", "xgboost": "XGBoost", "xgbregressor": "XGBoost",
    "lgbm": "LightGBM", "lightgbm": "LightGBM", "lgbmregressor": "LightGBM",
    "lstm": "Vanilla LSTM", "vanillalstm": "Vanilla LSTM",
    "residuallstm": "Residual LSTM",
    "lstmadapter": "Adapter LSTM", "adapterlstm": "Adapter LSTM",
    "meflstm": "Adapter LSTM", "emflstm": "Adapter LSTM",
    "meanembeddingforecastlstmwithadapter": "Adapter LSTM",
}
NAME_PATTERNS = [
    ("Adapter LSTM", r"(?:meanembeddingforecastlstmwithadapter|lstm[_ -]?adapter|adapter[_ -]?lstm|(?:mef|emf)[_ -]?lstm)"),
    ("Residual LSTM", r"residual[_ -]?lstm"),
    ("Vanilla LSTM", r"(?:vanilla[_ -]?lstm|lstm)"),
    ("XGBoost", r"(?:xgbregressor|xgboost|xgb)"),
    ("LightGBM", r"(?:lgbmregressor|lightgbm|lgbm)"),
]
FEATURE_ORDER = ["water_level", "rain", "wind", "reservoir", "flood_season"]


def canonical_model(value):
    key = re.sub(r"[^a-z0-9]", "", str(value).lower())
    return MODEL_ALIASES.get(key)


def canonical_features(value):
    if value is None or not str(value).strip():
        return None
    text = str(value).strip().lower()
    if text == "all":
        return "all"
    tokens = [token.strip() for token in re.split(r"\s*\+\s*", text)]
    tokens = ["water_level" if token == "water" else token for token in tokens]
    if any(not re.fullmatch(r"[a-z][a-z0-9_]*", token) for token in tokens):
        return None
    tokens = set(tokens)
    ordered = [token for token in FEATURE_ORDER if token in tokens]
    ordered += sorted(tokens - set(FEATURE_ORDER))
    return "+".join(ordered)


def parse_run_name(name):
    name = str(name or "").strip()
    for model, pattern in NAME_PATTERNS:
        match = re.match(rf"^{pattern}(?=$|[_ :\-])", name, flags=re.I)
        if not match:
            continue
        remainder = name[match.end():].lstrip("_ :-")
        resolution = re.search(r"(?:^|_)(?:10min|30min|1h)_", remainder, flags=re.I)
        if resolution:
            remainder = remainder[resolution.end():]
        remainder = re.sub(r"(?:_(?:two_stage|unfrozen|frozen))+$", "", remainder, flags=re.I)
        return model, canonical_features(remainder)
    return None, None


def run_identity(run):
    params, tags = run.data.params, run.data.tags
    name = tags.get("mlflow.runName") or getattr(run.info, "run_name", "") or ""
    model, feature_set = None, None
    model_source, feature_source = None, None
    for source, metadata in [("params", params), ("tags", tags)]:
        if model is None:
            for key in ["train_method", "model_type", "model_name", "model", "algorithm"]:
                candidate = canonical_model(metadata.get(key, ""))
                if candidate:
                    model, model_source = candidate, f"{source}.{key}"
                    break
        if feature_set is None:
            for key in ["feature_set", "lstm_feature_set", "features"]:
                candidate = canonical_features(metadata.get(key))
                if candidate:
                    feature_set, feature_source = candidate, f"{source}.{key}"
                    break
    fallback_model, fallback_features = parse_run_name(name)
    if model is None and fallback_model:
        model, model_source = fallback_model, "run_name"
    if feature_set is None and fallback_features:
        feature_set, feature_source = fallback_features, "run_name"
    return name, model, feature_set, model_source, feature_source


def finite_metric(metrics, key):
    try:
        value = float(metrics.get(key, np.nan))
        return value if np.isfinite(value) else np.nan
    except (TypeError, ValueError):
        return np.nan


RUN_COLUMNS = ["experiment_name", "run_id", "run_name", "model_type", "feature_set", "test_loss",
               "test_loss_scale", "dataset_dir", "resolution", "lifecycle_stage",
               "model_source", "feature_source", "start_time"]
records = []
for run in finished_runs:
    name, model, features, model_source, feature_source = run_identity(run)
    records.append({
        "experiment_name": experiment_by_id[run.info.experiment_id],
        "run_id": run.info.run_id, "run_name": name,
        "model_type": model, "feature_set": features,
        "test_loss": finite_metric(run.data.metrics, "test_loss"),
        "test_loss_scale": run.data.tags.get("test_loss_scale", "unknown"),
        "dataset_dir": run.data.params.get("dataset_dir", "unknown"),
        "resolution": run.data.params.get("horizon", "unknown"),
        "lifecycle_stage": run.info.lifecycle_stage,
        "model_source": model_source, "feature_source": feature_source,
        "start_time": run.info.start_time,
    })
all_runs_df = pd.DataFrame(records, columns=RUN_COLUMNS)
model_runs_df = all_runs_df.loc[all_runs_df["model_type"].isin(MODEL_TYPES)].copy()
unclassified_runs = all_runs_df.loc[all_runs_df["model_type"].isna()]
print(f"Run thuộc 5 nhóm model: {len(model_runs_df)}")
if not unclassified_runs.empty:
    print("Run không nhận diện được model (không đưa vào báo cáo):")
    display(unclassified_runs[["experiment_name", "run_id", "run_name"]])


Run thuộc 5 nhóm model: 116


In [4]:
GROUP_KEYS = ["experiment_name", "model_type", "feature_set"]
eligible_runs = model_runs_df.loc[
    model_runs_df["feature_set"].notna() & model_runs_df["test_loss"].notna()
    & model_runs_df["resolution"].isin(["1h", "unknown"])
].copy()
excluded_runs = model_runs_df.loc[~model_runs_df.index.isin(eligible_runs.index)]
best_runs = (eligible_runs.sort_values(
    [*GROUP_KEYS, "test_loss", "start_time", "run_id"],
    ascending=[True, True, True, True, False, True],
).drop_duplicates(GROUP_KEYS).reset_index(drop=True))
for experiment in EXPERIMENT_NAMES:
    print(f"=== {experiment}: run được chọn riêng cho mỗi model + feature set ===")
    display(best_runs.loc[best_runs.experiment_name.eq(experiment),
                         [*GROUP_KEYS, "run_name", "run_id", "test_loss"]].round(6))
if not excluded_runs.empty:
    print("Run không đủ thông tin hoặc không có độ phân giải 1h:")
    display(excluded_runs[["experiment_name", "run_id", "run_name", "test_loss", "resolution"]])

=== 1h: run được chọn riêng cho mỗi model + feature set ===


,experiment_name,model_type,feature_set,run_name,run_id,test_loss
0,1h,Adapter LSTM,all,lstm_adapter_1h_all_two_stage,3d0cf00162384df68ae69b81faf99341,0.416685
1,1h,Adapter LSTM,water_level,lstm_adapter_1h_water_level_unfrozen,ee746b6d99f24e7eab1c4c5b7f8458fc,0.136669
2,1h,Adapter LSTM,water_level+rain,lstm_adapter_1h_water_level+rain_two_stage,4909104e216149788b094e3acfc4a439,0.348901
3,1h,LightGBM,all,lgbm_1h_all,3b3b88e73c9f4e4289d6a4bd04f64fce,18914.299682
4,1h,LightGBM,water_level,lgbm_1h_water_level,73470359f6ec414abe8fb6a6988aa0d4,11130.172669
5,1h,LightGBM,water_level+rain,lgbm_1h_water_level+rain,86e2c54a3d5b41f09ec85f2791899ae5,14400.617792
6,1h,Residual LSTM,all,residual_lstm_1h_all,f4a41c88a1374c97bd35599a899d813e,0.180603
7,1h,Residual LSTM,water_level,residual_lstm_1h_water_level,4335acb3a3704626a782ea3635ce4204,0.093164
8,1h,Residual LSTM,water_level+rain,residual_lstm_1h_water_level+rain,306db755bc50422c8ff63af4258ee759,0.182173
9,1h,Vanilla LSTM,all,lstm_1h_all,1be43fe5bedd4e1aab79e28d4228414d,0.499385


=== 1h_top4: run được chọn riêng cho mỗi model + feature set ===


,experiment_name,model_type,feature_set,run_name,run_id,test_loss
15,1h_top4,Adapter LSTM,water_level+flood_season,lstm_adapter_1h_water_level+flood_season_two_stage,4412e24ec4f2429dbb044a2154c3d972,0.193235
16,1h_top4,Adapter LSTM,water_level+rain+flood_season,lstm_adapter_1h_water_level+rain+flood_season_two_stage,3dbb0f8715f24548b33a35d669839191,0.328886
17,1h_top4,LightGBM,water_level+flood_season,lgbm_1h_water_level+flood_season,8b33169fa682411198da2fff2e7718e4,11150.010719
18,1h_top4,LightGBM,water_level+rain,lgbm_1h_water_level+rain,df2039b5604846f78f055dd5430162e2,23415.666684
19,1h_top4,LightGBM,water_level+rain+flood_season,lgbm_1h_water_level+rain+flood_season,f24da39ef1514622bb918adaff72db27,24210.366562
20,1h_top4,Residual LSTM,water_level,base_24hidden_residual_lstm_1h_water_level,e8f8c73778ee49a5a31f902647a7f65a,0.104666
21,1h_top4,Residual LSTM,water_level+flood_season,residual_lstm_1h_water_level+flood_season,0c1742067b634c84883d48f0ee5278ad,0.102812
22,1h_top4,Residual LSTM,water_level+rain,residual_lstm_1h_water_level+rain,859a14c07032471eb656d8a8e13ae0d3,0.104992
23,1h_top4,Residual LSTM,water_level+rain+flood_season,BEST_32_96SEQLEN_residual_lstm_1h_water_level+rain+flood_season,6472d6d3d42f44c3a29f730b4d0ca828,0.091426
24,1h_top4,Vanilla LSTM,water_level+flood_season,BEST_lstm_1h_water_level+flood_season,5698bf6a05c24fc1a5fd40382998c54d,0.118912


## 3. Metric theo mốc dự báo và các bảng báo cáo

Mỗi run được chọn có đủ 5 dòng, kể cả khi một số metric chưa được log. Giá trị thiếu được để NaN. Các bảng chỉ làm tròn khi hiển thị; file export giữ độ chính xác số liệu.


In [5]:
PROVENANCE = ["experiment_name", "run_id", "run_name", "test_loss_scale", "dataset_dir", "resolution", "lifecycle_stage"]
LONG_COLUMNS = ["model_type", "feature_set", "horizon_min", "test_loss", *METRICS, *PROVENANCE]
horizon_records = []
for selected in best_runs.to_dict("records"):
    metrics = runs_by_id[selected["run_id"]].data.metrics
    for horizon in HORIZONS:
        row = {key: selected[key] for key in ["model_type", "feature_set", "test_loss", *PROVENANCE]}
        row["horizon_min"] = horizon
        for metric in METRICS:
            row[metric] = finite_metric(metrics, f"test/target_water_level_plus_{horizon}m/{metric}")
        horizon_records.append(row)
by_horizon = pd.DataFrame(horizon_records, columns=LONG_COLUMNS)
print("Metric từng horizon — có experiment_name để phân biệt hai dataset:")
display(by_horizon[["experiment_name", "model_type", "feature_set", "horizon_min", "test_loss", *METRICS]].round(3))


def horizon_pivot(frame, metric):
    if frame.empty:
        return pd.DataFrame(columns=["Experiment", "Model", "Feature set", *[f"{h}m" for h in HORIZONS]])
    table = frame.pivot(index=["experiment_name", "model_type", "feature_set"], columns="horizon_min", values=metric)
    table = table.reindex(columns=HORIZONS).rename(columns={h: f"{h}m" for h in HORIZONS}).reset_index()
    table.columns.name = None
    return table.rename(columns={"experiment_name": "Experiment", "model_type": "Model", "feature_set": "Feature set"})


PIVOT_LABELS = {
    "rmse": "RMSE", "mae": "MAE", "nse": "NSE", "kge": "KGE",
    "peak_absolute_error": "Peak Error", "peak_timing_error_minutes": "Peak Timing",
}
pivot_tables = {metric: horizon_pivot(by_horizon, metric) for metric in METRICS}
for experiment in EXPERIMENT_NAMES:
    print(f"=== {experiment}: toàn bộ Test ===")
    for metric, table in pivot_tables.items():
        unit = "phút" if metric == "peak_timing_error_minutes" else ("mm" if metric in ["rmse", "mae", "peak_absolute_error"] else "không đơn vị")
        print(f"{PIVOT_LABELS[metric]} ({unit})")
        display(table.loc[table.Experiment.eq(experiment)].round(3))

Metric từng horizon — có experiment_name để phân biệt hai dataset:


,experiment_name,model_type,feature_set,horizon_min,test_loss,rmse,mae,nse,kge,peak_absolute_error,peak_timing_error_minutes
0,1h,Adapter LSTM,all,60,0.417,135.737,112.395,0.151,0.343,527.511,120.0
1,1h,Adapter LSTM,all,180,0.417,148.850,125.437,-0.022,0.232,556.212,0.0
2,1h,Adapter LSTM,all,360,0.417,155.909,131.187,-0.120,0.181,591.807,180.0
3,1h,Adapter LSTM,all,720,0.417,154.589,126.164,-0.080,0.146,590.780,4320.0
4,1h,Adapter LSTM,all,1440,0.417,154.526,127.205,-0.060,0.198,601.445,1260.0
5,1h,Adapter LSTM,water_level,60,0.137,41.392,26.503,0.916,0.768,224.789,0.0
6,1h,Adapter LSTM,water_level,180,0.137,57.935,42.332,0.835,0.749,221.626,60.0
7,1h,Adapter LSTM,water_level,360,0.137,87.651,63.332,0.623,0.607,388.190,60.0
8,1h,Adapter LSTM,water_level,720,0.137,105.649,75.744,0.462,0.477,384.329,780.0
9,1h,Adapter LSTM,water_level,1440,0.137,113.683,81.484,0.386,0.412,346.398,1440.0


=== 1h: toàn bộ Test ===
RMSE (mm)


,Experiment,Model,Feature set,60m,180m,360m,720m,1440m
0,1h,Adapter LSTM,all,135.737,148.850,155.909,154.589,154.526
1,1h,Adapter LSTM,water_level,41.392,57.935,87.651,105.649,113.683
2,1h,Adapter LSTM,water_level+rain,121.257,130.274,141.157,147.565,144.683
3,1h,LightGBM,all,42.347,83.424,128.125,223.407,139.612
4,1h,LightGBM,water_level,34.116,84.687,117.284,122.001,136.657
5,1h,LightGBM,water_level+rain,44.771,92.188,137.530,154.724,136.551
6,1h,Residual LSTM,all,34.491,92.883,137.622,100.453,99.800
7,1h,Residual LSTM,water_level,19.350,50.589,75.825,82.019,98.438
8,1h,Residual LSTM,water_level+rain,34.384,92.569,137.794,101.200,101.235
9,1h,Vanilla LSTM,all,161.942,162.454,162.482,166.186,168.569


MAE (mm)


...

,Experiment,Model,Feature set,60m,180m,360m,720m,1440m
15,1h_top4,Adapter LSTM,water_level+flood_season,0.798,0.596,0.358,0.389,0.463
16,1h_top4,Adapter LSTM,water_level+rain+flood_season,0.521,0.358,0.238,0.286,0.301
17,1h_top4,LightGBM,water_level+flood_season,0.959,0.776,0.559,0.592,0.599
18,1h_top4,LightGBM,water_level+rain,0.956,0.767,0.541,0.443,-0.018
19,1h_top4,LightGBM,water_level+rain+flood_season,0.954,0.758,0.537,0.436,-0.061
20,1h_top4,Residual LSTM,water_level,0.973,0.878,0.766,0.752,0.625
21,1h_top4,Residual LSTM,water_level+flood_season,0.953,0.852,0.735,0.717,0.610
22,1h_top4,Residual LSTM,water_level+rain,0.977,0.885,0.760,0.804,0.724
23,1h_top4,Residual LSTM,water_level+rain+flood_season,0.983,0.892,0.779,0.801,0.714
24,1h_top4,Vanilla LSTM,water_level+flood_season,0.818,0.766,0.633,0.603,0.495


Peak Error (mm)


,Experiment,Model,Feature set,60m,180m,360m,720m,1440m
15,1h_top4,Adapter LSTM,water_level+flood_season,381.453,423.350,514.393,542.383,563.268
16,1h_top4,Adapter LSTM,water_level+rain+flood_season,493.796,545.375,591.520,551.094,507.880
17,1h_top4,LightGBM,water_level+flood_season,6.169,48.400,174.276,414.264,525.644
18,1h_top4,LightGBM,water_level+rain,116.357,340.132,797.541,1237.216,1119.335
19,1h_top4,LightGBM,water_level+rain+flood_season,111.111,337.166,768.930,1260.208,1179.181
20,1h_top4,Residual LSTM,water_level,34.332,113.677,144.385,117.456,151.684
21,1h_top4,Residual LSTM,water_level+flood_season,35.864,112.048,163.558,120.032,117.894
22,1h_top4,Residual LSTM,water_level+rain,32.801,98.854,104.926,59.591,3.463
23,1h_top4,Residual LSTM,water_level+rain+flood_season,29.824,95.523,146.755,44.794,53.857
24,1h_top4,Vanilla LSTM,water_level+flood_season,212.663,245.722,314.339,324.403,386.562


Peak Timing (phút)


,Experiment,Model,Feature set,60m,180m,360m,720m,1440m
15,1h_top4,Adapter LSTM,water_level+flood_season,60.0,840.0,1020.0,1380.0,120.0
16,1h_top4,Adapter LSTM,water_level+rain+flood_season,2880.0,3000.0,9840.0,3540.0,5820.0
17,1h_top4,LightGBM,water_level+flood_season,180.0,300.0,360.0,720.0,1380.0
18,1h_top4,LightGBM,water_level+rain,180.0,300.0,480.0,720.0,1260.0
19,1h_top4,LightGBM,water_level+rain+flood_season,180.0,300.0,480.0,840.0,1260.0
20,1h_top4,Residual LSTM,water_level,60.0,180.0,420.0,720.0,1440.0
21,1h_top4,Residual LSTM,water_level+flood_season,60.0,180.0,420.0,720.0,1440.0
22,1h_top4,Residual LSTM,water_level+rain,60.0,180.0,480.0,720.0,1440.0
23,1h_top4,Residual LSTM,water_level+rain+flood_season,60.0,60.0,60.0,780.0,1440.0
24,1h_top4,Vanilla LSTM,water_level+flood_season,60.0,240.0,420.0,780.0,1440.0


In [6]:
def mean_if_available(values):
    valid = pd.to_numeric(values, errors="coerce").dropna()
    return float(valid.mean()) if not valid.empty else np.nan


summary_records = []
for selected in best_runs.to_dict("records"):
    horizons = by_horizon.loc[by_horizon["run_id"].eq(selected["run_id"])]
    row = {key: selected[key] for key in ["model_type", "feature_set", "test_loss", *PROVENANCE]}
    for metric in METRICS:
        row[f"mean_{metric}"] = mean_if_available(horizons[metric])
    summary_records.append(row)
SUMMARY_COLUMNS = ["model_type", "feature_set", "test_loss", *[f"mean_{metric}" for metric in METRICS], *PROVENANCE]
summary = pd.DataFrame(summary_records, columns=SUMMARY_COLUMNS).sort_values(
    ["experiment_name", "model_type", "test_loss", "feature_set"], na_position="last"
).reset_index(drop=True)
SUMMARY_LABELS = {
    "experiment_name": "Experiment", "model_type": "Model", "feature_set": "Feature set", "test_loss": "Test loss",
    "mean_rmse": "Mean RMSE", "mean_mae": "Mean MAE", "mean_nse": "Mean NSE", "mean_kge": "Mean KGE",
    "mean_peak_absolute_error": "Mean Peak Absolute Error",
    "mean_peak_timing_error_minutes": "Mean Peak Timing Error",
}
summary_report = summary[list(SUMMARY_LABELS)].rename(columns=SUMMARY_LABELS)
print("Summary (trung bình các horizon có metric, không thay thế kết quả từng horizon):")
display(summary_report.round(3))
best_feature_sets = summary.drop_duplicates(["experiment_name", "model_type"])
print("Feature set có test_loss thấp nhất của từng model:")
for row in best_feature_sets.itertuples():
    print(f"{row.model_type} -> {row.feature_set} | test_loss={row.test_loss:.6g} | experiment={row.experiment_name}")
missing_models = sorted(set(MODEL_TYPES) - set(summary["model_type"]))
if missing_models:
    print("Model chưa có run đủ dữ liệu để chọn:", missing_models)
print("Số giá trị metric thiếu trong bảng từng horizon:")
display(by_horizon[METRICS].isna().sum().rename("missing_count").to_frame())

Summary (trung bình các horizon có metric, không thay thế kết quả từng horizon):


,Experiment,Model,Feature set,Test loss,Mean RMSE,Mean MAE,Mean NSE,Mean KGE,Mean Peak Absolute Error,Mean Peak Timing Error
0,1h,Adapter LSTM,water_level,0.137,81.262,57.879,0.644,0.603,313.066,468.0
1,1h,Adapter LSTM,water_level+rain,0.349,136.987,100.684,0.142,0.237,625.164,2688.0
2,1h,Adapter LSTM,all,0.417,149.922,124.478,-0.026,0.220,573.551,1176.0
3,1h,LightGBM,water_level,11130.173,98.949,68.583,0.445,0.681,227.663,600.0
4,1h,LightGBM,water_level+rain,14400.618,113.153,84.441,0.282,0.619,235.212,648.0
5,1h,LightGBM,all,18914.300,123.383,85.221,0.058,0.533,413.108,672.0
6,1h,Residual LSTM,water_level,0.093,65.244,45.493,0.762,0.803,107.533,588.0
7,1h,Residual LSTM,all,0.181,93.050,65.352,0.533,0.755,21.369,552.0
8,1h,Residual LSTM,water_level+rain,0.182,93.437,65.466,0.530,0.737,38.798,552.0
9,1h,Vanilla LSTM,water_level,0.132,79.310,55.219,0.680,0.711,259.664,684.0


Feature set có test_loss thấp nhất của từng model:
Adapter LSTM -> water_level | test_loss=0.136669 | experiment=1h
LightGBM -> water_level | test_loss=11130.2 | experiment=1h
Residual LSTM -> water_level | test_loss=0.0931639 | experiment=1h
Vanilla LSTM -> water_level | test_loss=0.131574 | experiment=1h
XGBoost -> water_level | test_loss=10441.8 | experiment=1h
Adapter LSTM -> water_level+flood_season | test_loss=0.193235 | experiment=1h_top4
LightGBM -> water_level+flood_season | test_loss=11150 | experiment=1h_top4
Residual LSTM -> water_level+rain+flood_season | test_loss=0.0914265 | experiment=1h_top4
Vanilla LSTM -> water_level+flood_season | test_loss=0.118912 | experiment=1h_top4
XGBoost -> water_level+flood_season | test_loss=11327.5 | experiment=1h_top4
Số giá trị metric thiếu trong bảng từng horizon:


,missing_count
rmse,0
mae,0
nse,0
kge,0
peak_absolute_error,0
peak_timing_error_minutes,0


## 4. Tính metric riêng cho mùa lũ và ngoài mùa lũ

Ưu tiên metric theo mùa đã log. Nếu chưa có, tìm artifact chứa **timestamp + observed + predicted**; nếu không có thì dùng **model và bộ chuẩn hóa đã lưu**, chạy inference trên đúng Test của dataset tương ứng. Không fit lại scaler và không train model.

Trước khi dùng inference, đối chiếu RMSE/MAE trên toàn bộ Test với MLflow. Nếu không khớp, không xuất metric mùa tính từ bộ dữ liệu đó. Bảng kiểm tra ghi rõ nguồn và lỗi để tránh dùng nhầm dữ liệu. Giá trị không tính được để NaN.

Mỗi nhóm mùa được xác định bằng timestamp tại t, đổi UTC sang giờ Việt Nam. Đỉnh được tính trên toàn nhóm đã chọn, do đó sai lệch thời gian đỉnh có thể lớn nếu model chọn một đợt khác làm đỉnh cao nhất. Bảng có số mẫu và khoảng thời gian để người đọc biết phạm vi đánh giá.

In [7]:
import hashlib
import json
import sys
import tempfile
import warnings
import joblib
import torch
from torch.utils.data import DataLoader

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.ai.dataset import HydrologyDataset
from src.ai.metrics import rmse, mae, nse, kge, peak_metrics

PERIODS = ["all", "flood_season", "non_flood_season"]
PERIOD_LABELS = {"all": "Toàn bộ Test", "flood_season": "Mùa lũ (tháng 9–12)",
                 "non_flood_season": "Ngoài mùa lũ"}
DATASET_DIRS = {"1h": PROJECT_ROOT / "train_data/1h",
                "1h_top4": PROJECT_ROOT / "train_data/1h_top4"}
# CPU mặc định để notebook không phụ thuộc GPU; chỉ inference, không fit/train.
INFERENCE_DEVICE = "cpu"
INFERENCE_BATCH_SIZE = 128
torch.set_num_threads(4)
artifact_tmp = tempfile.TemporaryDirectory(prefix="hydrology-metrics-")
local_model_paths = {}
for metadata_path in (PROJECT_ROOT / "mlartifacts").glob("*/models/*/artifacts/MLmodel"):
    local_model_paths[metadata_path.parent.parent.name] = metadata_path.parent


def run_artifact(run, artifact_path):
    """Read existing local artifact, otherwise download to a temporary directory."""
    local = PROJECT_ROOT / "mlartifacts" / run.info.experiment_id / run.info.run_id / "artifacts" / artifact_path
    if local.is_file():
        return local
    return Path(client.download_artifacts(run.info.run_id, artifact_path, artifact_tmp.name))


def model_path(uri):
    model_id = uri.removeprefix("models:/")
    if model_id in local_model_paths:
        return local_model_paths[model_id]
    return Path(mlflow.artifacts.download_artifacts(
        artifact_uri=uri, dst_path=artifact_tmp.name, tracking_uri=tracking_uri))


def list_run_files(run, path=""):
    result = []
    for item in client.list_artifacts(run.info.run_id, path):
        if item.is_dir:
            result.extend(list_run_files(run, item.path))
        else:
            result.append(item.path)
    return result


def prediction_artifact(run, files):
    """Accept only explicit long-form predictions with original-unit metadata."""
    for path in files:
        if not re.search(r"predictions?|evaluation", path, re.I) or not path.endswith(".csv"):
            continue
        frame = pd.read_csv(run_artifact(run, path))
        aliases = {"timestamp_utc": "timestamp", "y_true": "observed", "y_pred": "predicted"}
        frame = frame.rename(columns={k: v for k, v in aliases.items() if v not in frame.columns})
        required = {"timestamp", "horizon_min", "observed", "predicted"}
        if required.issubset(frame.columns):
            return frame[list(required)].copy(), f"artifact:{path}"
    return None, None


def evaluation_data(selected, run):
    expected = DATASET_DIRS[selected["experiment_name"]]
    configured = run.data.params.get("dataset_dir")
    directory = Path(configured) if configured else expected
    if not directory.is_absolute():
        directory = PROJECT_ROOT / directory
    if directory.resolve() != expected.resolve():
        raise ValueError(f"dataset_dir không khớp experiment: {directory}")
    path = directory / "test.csv"
    # Verify the actual Test file if the run recorded a manifest hash.
    manifest_path = PROJECT_ROOT / "mlartifacts" / run.info.experiment_id / run.info.run_id / "artifacts/dataset/manifest.json"
    if manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        expected_hash = manifest.get("split_sha256", {}).get("test")
        if expected_hash and hashlib.sha256(path.read_bytes()).hexdigest() != expected_hash:
            raise ValueError("Test CSV đã thay đổi so với manifest của run")
    frame = pd.read_csv(path)
    if "timestamp_utc" not in frame:
        raise ValueError("Test thiếu timestamp_utc")
    ts = pd.to_datetime(frame.timestamp_utc, utc=True, errors="raise")
    if ts.duplicated().any() or not ts.is_monotonic_increasing:
        raise ValueError("Timestamp Test trùng hoặc không tăng dần")
    return frame, str(path), hashlib.sha256(path.read_bytes()).hexdigest()


def infer_predictions(selected, run, files):
    frame, path, sha = evaluation_data(selected, run)
    target_cols = [f"target_water_level_plus_{h}m" for h in HORIZONS]
    if selected["model_type"] in ["XGBoost", "LightGBM"]:
        features = json.loads(run_artifact(run, "features.json").read_text())["feature_columns"]
        frame = frame.dropna(subset=target_cols).reset_index(drop=True)
        observed = frame[target_cols].to_numpy(float)
        predicted = np.empty_like(observed)
        for j, target in enumerate(target_cols):
            uri = run.data.tags.get(f"model_uri/{target}")
            if not uri:
                raise ValueError(f"Không có model_uri/{target}")
            if selected["model_type"] == "XGBoost":
                import mlflow.xgboost
                model = mlflow.xgboost.load_model(str(model_path(uri)))
            else:
                import mlflow.lightgbm
                model = mlflow.lightgbm.load_model(str(model_path(uri)))
            predicted[:, j] = model.predict(frame[features])
        timestamps = frame.timestamp_utc
    else:
        uri = run.data.tags.get("model_uri/best_model")
        if not uri:
            raise ValueError("Không có model_uri/best_model; không dùng checkpoint local có thể đã bị ghi đè")
        directory = model_path(uri)
        preprocessing = joblib.load(directory / "extra_files/preprocessing.joblib")
        target_cols = preprocessing["target_columns"]
        features = preprocessing["feature_columns"]
        requires_level = "water_level_depth" in features
        dataset = HydrologyDataset(
            frame, feature_cols=features, target_cols=target_cols,
            feature_scaler=preprocessing["feature_scaler"],
            target_scaler=preprocessing["target_scaler"],
            seq_len=int(preprocessing["sequence_length"]), require_current_level=requires_level,
        )
        import mlflow.pytorch
        model = mlflow.pytorch.load_model(str(directory), map_location=INFERENCE_DEVICE)
        # A pickle can resolve src.models to today's code. Use the class saved
        # alongside this model so older runs execute their original forward().
        archived_source = directory / "code/src/ai/models.py"
        if not archived_source.is_file():
            archived_source = directory / "code/src/models.py"
        if archived_source.is_file():
            module_name = "hydrology_saved_" + run.info.run_id
            spec = importlib.util.spec_from_file_location(module_name, archived_source)
            archived_module = importlib.util.module_from_spec(spec)
            sys.modules[module_name] = archived_module
            spec.loader.exec_module(archived_module)
            saved_class = getattr(archived_module, type(model).__name__, None)
            if saved_class is not None:
                model.__class__ = saved_class
        model.to(INFERENCE_DEVICE).eval()
        batches = []
        with torch.inference_mode():
            for inputs, _ in DataLoader(dataset, batch_size=INFERENCE_BATCH_SIZE, shuffle=False, drop_last=False):
                batches.append(model(inputs.to(INFERENCE_DEVICE)).cpu().numpy())
        if not batches:
            raise ValueError("Không có mẫu Test hợp lệ")
        predicted = preprocessing["target_scaler"].inverse_transform(np.concatenate(batches))
        # Match trainer's float32 transform/inverse-transform of ground truth.
        observed = preprocessing["target_scaler"].inverse_transform(dataset.y[dataset.end_indices])
        timestamps = dataset.timestamps
    rows = []
    for j, target in enumerate(target_cols):
        horizon = int(re.search(r"plus_(\d+)m", target).group(1))
        if horizon in HORIZONS:
            rows.append(pd.DataFrame({"timestamp": pd.to_datetime(timestamps, utc=True).to_numpy(),
                                      "horizon_min": horizon, "observed": observed[:, j],
                                      "predicted": predicted[:, j]}))
    return pd.concat(rows, ignore_index=True), f"inference:{path}", sha


def period_mask(timestamps, period):
    months = pd.to_datetime(timestamps, utc=True).dt.tz_convert("Asia/Ho_Chi_Minh").dt.month
    flood = months.isin([9, 10, 11, 12])
    return pd.Series(True, index=timestamps.index) if period == "all" else (flood if period == "flood_season" else ~flood)


def subset_metrics(frame):
    valid = np.isfinite(frame.observed) & np.isfinite(frame.predicted)
    frame = frame.loc[valid].sort_values("timestamp")
    result = {metric: np.nan for metric in METRICS}
    if not frame.empty:
        obs, pred = frame.observed.to_numpy(), frame.predicted.to_numpy()
        result.update(rmse=rmse(obs, pred), mae=mae(obs, pred), nse=nse(obs, pred), kge=kge(obs, pred))
        result.update({k: np.nan if v is None else v for k, v in
                       peak_metrics(obs, pred, timestamps=frame.timestamp).items()})
    return {**result, "sample_count": len(frame),
            "start_time": frame.timestamp.min() if not frame.empty else pd.NaT,
            "end_time": frame.timestamp.max() if not frame.empty else pd.NaT}


def logged_period_metric(metrics, period, horizon, metric):
    target = f"target_water_level_plus_{horizon}m"
    keys = [f"test/{target}/{metric}"] if period == "all" else [
        f"test/{period}/{target}/{metric}", f"test/{target}/{period}/{metric}",
        f"test_{period}/{target}/{metric}", f"test/{target}/{metric}/{period}"]
    for key in keys:
        value = finite_metric(metrics, key)
        if pd.notna(value):
            return value, key
    return np.nan, None


def validate_replay(predictions, metrics):
    checks = []
    for horizon in HORIZONS:
        values = subset_metrics(predictions.loc[predictions.horizon_min.eq(horizon)])
        for metric in ["rmse", "mae"]:
            logged = finite_metric(metrics, f"test/target_water_level_plus_{horizon}m/{metric}")
            if pd.notna(logged):
                checks.append(np.isclose(values[metric], logged, rtol=1e-4, atol=0.005))
    if not checks:
        raise ValueError("Không có RMSE/MAE toàn tập để xác minh đơn vị và mẫu Test")
    if not all(checks):
        raise ValueError("Inference không khớp RMSE/MAE đã log; không dùng kết quả này để tính theo mùa")

In [8]:
period_records, audit_records, predictions_by_run = [], [], {}
for selected in best_runs.to_dict("records"):
    run = runs_by_id[selected["run_id"]]
    logged = run.data.metrics
    predictions, source, sha, error = None, "logged", None, None
    needs_predictions = any(pd.isna(logged_period_metric(logged, period, h, metric)[0])
                            for period in PERIODS[1:] for h in HORIZONS for metric in METRICS)
    if needs_predictions:
        print(f"Đọc Test: {selected['experiment_name']} | {selected['model_type']} | {selected['feature_set']}")
        try:
            files = list_run_files(run)
            predictions, source = prediction_artifact(run, files)
            if predictions is None:
                predictions, source, sha = infer_predictions(selected, run, files)
            predictions["timestamp"] = pd.to_datetime(predictions.timestamp, utc=True, errors="raise")
            if predictions.duplicated(["timestamp", "horizon_min"]).any():
                raise ValueError("Prediction trùng timestamp + horizon")
            validate_replay(predictions, logged)
            predictions_by_run[selected["run_id"]] = predictions
        except Exception as exc:
            error = f"{type(exc).__name__}: {exc}"
            print("  Chưa tính được metric mùa:", error)
            predictions = None
    audit_records.append({**{k: selected[k] for k in [*GROUP_KEYS, "run_id"]},
                          "source": source, "replay_verified": predictions is not None,
                          "test_sha256": sha, "error": error})
    for period in PERIODS:
        for horizon in HORIZONS:
            row = {k: selected[k] for k in [*GROUP_KEYS, "run_id", "test_loss"]}
            row.update(period=period, horizon_min=horizon, sample_count=np.nan,
                       start_time=pd.NaT, end_time=pd.NaT)
            calculated = None
            if predictions is not None:
                subset = predictions.loc[predictions.horizon_min.eq(horizon)]
                calculated = subset_metrics(subset.loc[period_mask(subset.timestamp, period)])
                row.update({k: calculated[k] for k in ["sample_count", "start_time", "end_time"]})
            metric_sources = set()
            for metric in METRICS:
                value, key = logged_period_metric(logged, period, horizon, metric)
                if key:
                    metric_sources.add("logged")
                elif calculated is not None:
                    value = calculated[metric]
                    metric_sources.add(source)
                else:
                    metric_sources.add("unavailable")
                row[metric] = value
            row["metric_source"] = "; ".join(sorted(metric_sources))
            period_records.append(row)
by_period = pd.DataFrame(period_records)
evaluation_audit = pd.DataFrame(audit_records)
print("Kiểm tra inference so với metric MLflow:")
display(evaluation_audit[[*GROUP_KEYS, "replay_verified", "error"]])
if evaluation_audit.error.notna().any():
    print("Các dòng không đủ dữ liệu giữ NaN; không suy ra metric mùa từ metric toàn tập.")
print("Số mẫu và khoảng thời gian từng mùa (thời gian hiển thị UTC):")
display(by_period.loc[by_period.horizon_min.eq(60),
                     [*GROUP_KEYS, "period", "sample_count", "start_time", "end_time"]])

Đọc Test: 1h | Adapter LSTM | all
Đọc Test: 1h | Adapter LSTM | water_level
Đọc Test: 1h | Adapter LSTM | water_level+rain
Đọc Test: 1h | LightGBM | all
Đọc Test: 1h | LightGBM | water_level
Đọc Test: 1h | LightGBM | water_level+rain
Đọc Test: 1h | Residual LSTM | all
Đọc Test: 1h | Residual LSTM | water_level
Đọc Test: 1h | Residual LSTM | water_level+rain
Đọc Test: 1h | Vanilla LSTM | all
Đọc Test: 1h | Vanilla LSTM | water_level
Đọc Test: 1h | Vanilla LSTM | water_level+rain
Đọc Test: 1h | XGBoost | all
Đọc Test: 1h | XGBoost | water_level
Đọc Test: 1h | XGBoost | water_level+rain
Đọc Test: 1h_top4 | Adapter LSTM | water_level+flood_season
Đọc Test: 1h_top4 | Adapter LSTM | water_level+rain+flood_season
Đọc Test: 1h_top4 | LightGBM | water_level+flood_season
Đọc Test: 1h_top4 | LightGBM | water_level+rain
Đọc Test: 1h_top4 | LightGBM | water_level+rain+flood_season
Đọc Test: 1h_top4 | Residual LSTM | water_level
Đọc Test: 1h_top4 | Residual LSTM | water_level+flood_season
Đọc Test: 

,experiment_name,model_type,feature_set,replay_verified,error
0,1h,Adapter LSTM,all,True,None
1,1h,Adapter LSTM,water_level,True,None
2,1h,Adapter LSTM,water_level+rain,True,None
3,1h,LightGBM,all,True,None
4,1h,LightGBM,water_level,True,None
5,1h,LightGBM,water_level+rain,True,None
6,1h,Residual LSTM,all,True,None
7,1h,Residual LSTM,water_level,True,None
8,1h,Residual LSTM,water_level+rain,True,None
9,1h,Vanilla LSTM,all,True,None


Số mẫu và khoảng thời gian từng mùa (thời gian hiển thị UTC):


,experiment_name,model_type,feature_set,period,sample_count,start_time,end_time
0,1h,Adapter LSTM,all,all,1004,2026-08-18 09:00:00+00:00,2026-09-29 04:00:00+00:00
5,1h,Adapter LSTM,all,flood_season,684,2026-08-31 17:00:00+00:00,2026-09-29 04:00:00+00:00
10,1h,Adapter LSTM,all,non_flood_season,320,2026-08-18 09:00:00+00:00,2026-08-31 16:00:00+00:00
15,1h,Adapter LSTM,water_level,all,1100,2026-08-14 09:00:00+00:00,2026-09-29 04:00:00+00:00
20,1h,Adapter LSTM,water_level,flood_season,684,2026-08-31 17:00:00+00:00,2026-09-29 04:00:00+00:00
25,1h,Adapter LSTM,water_level,non_flood_season,416,2026-08-14 09:00:00+00:00,2026-08-31 16:00:00+00:00
30,1h,Adapter LSTM,water_level+rain,all,1004,2026-08-18 09:00:00+00:00,2026-09-29 04:00:00+00:00
35,1h,Adapter LSTM,water_level+rain,flood_season,684,2026-08-31 17:00:00+00:00,2026-09-29 04:00:00+00:00
40,1h,Adapter LSTM,water_level+rain,non_flood_season,320,2026-08-18 09:00:00+00:00,2026-08-31 16:00:00+00:00
45,1h,LightGBM,all,all,1147,2026-08-12 10:00:00+00:00,2026-09-29 04:00:00+00:00


In [9]:
def comparison_table(frame):
    keys = [*GROUP_KEYS, "period"]
    wide = frame.pivot(index=keys, columns="horizon_min", values="rmse").reindex(columns=HORIZONS)
    wide = wide.rename(columns={h: f"{h}m RMSE (mm)" for h in HORIZONS})
    means = frame.groupby(keys, sort=False)[["mae", "nse", "kge", "peak_absolute_error",
                                           "peak_timing_error_minutes"]].agg(mean_if_available)
    means = means.rename(columns={"mae": "Mean MAE (mm)", "nse": "Mean NSE", "kge": "Mean KGE",
                                 "peak_absolute_error": "Mean Peak Error (mm)",
                                 "peak_timing_error_minutes": "Mean Peak Timing (phút)"})
    return wide.join(means).reset_index().rename(columns={"experiment_name": "Experiment",
        "model_type": "Model", "feature_set": "Feature set", "period": "Period"})

period_comparison = comparison_table(by_period)
period_tables = {}
for period in PERIODS:
    table = period_comparison.loc[period_comparison.Period.eq(period)].copy()
    period_tables[period] = table
    print(f"=== {PERIOD_LABELS[period]} — 1 đơn vị RMSE = 1 mm ===")
    for experiment in EXPERIMENT_NAMES:
        print(f"Dataset: {experiment}")
        display(table.loc[table.Experiment.eq(experiment)].round(3))
print("=== Bảng so sánh gộp hai experiment và ba nhóm thời gian ===")
display(period_comparison.round(3))

# Đối chiếu trực tiếp các feature set có ở cả hai experiment; không lẫn các run.
rmse_comparison = by_period.pivot(index=["model_type", "feature_set", "period", "horizon_min"],
                                 columns="experiment_name", values="rmse").reindex(columns=EXPERIMENT_NAMES)
rmse_comparison["top4_minus_1h_mm"] = rmse_comparison["1h_top4"] - rmse_comparison["1h"]
rmse_comparison = rmse_comparison.reset_index()
print("Chênh lệch RMSE của cùng model + feature set: số âm nghĩa là top4 có RMSE thấp hơn.")
display(rmse_comparison.dropna(subset=EXPERIMENT_NAMES).round(3))

# So sánh best feature set của mỗi model trong mỗi experiment theo đúng quy tắc test_loss.
best_feature_ids = set(best_feature_sets.run_id)
best_feature_periods = by_period.loc[by_period.run_id.isin(best_feature_ids)]
print("Best feature set của từng model trong mỗi dataset (không hard-code kết quả):")
display(comparison_table(best_feature_periods).round(3))

=== Toàn bộ Test — 1 đơn vị RMSE = 1 mm ===
Dataset: 1h


,Experiment,Model,Feature set,Period,60m RMSE (mm),180m RMSE (mm),360m RMSE (mm),720m RMSE (mm),1440m RMSE (mm),Mean MAE (mm),Mean NSE,Mean KGE,Mean Peak Error (mm),Mean Peak Timing (phút)
0,1h,Adapter LSTM,all,all,135.737,148.850,155.909,154.589,154.526,124.478,-0.026,0.220,573.551,1176.0
3,1h,Adapter LSTM,water_level,all,41.392,57.935,87.651,105.649,113.683,57.879,0.644,0.603,313.066,468.0
6,1h,Adapter LSTM,water_level+rain,all,121.257,130.274,141.157,147.565,144.683,100.684,0.142,0.237,625.164,2688.0
9,1h,LightGBM,all,all,42.347,83.424,128.125,223.407,139.612,85.221,0.058,0.533,413.108,672.0
12,1h,LightGBM,water_level,all,34.116,84.687,117.284,122.001,136.657,68.583,0.445,0.681,227.663,600.0
15,1h,LightGBM,water_level+rain,all,44.771,92.188,137.530,154.724,136.551,84.441,0.282,0.619,235.212,648.0
18,1h,Residual LSTM,all,all,34.491,92.883,137.622,100.453,99.800,65.352,0.533,0.755,21.369,552.0
21,1h,Residual LSTM,water_level,all,19.350,50.589,75.825,82.019,98.438,45.493,0.762,0.803,107.533,588.0
24,1h,Residual LSTM,water_level+rain,all,34.384,92.569,137.794,101.200,101.235,65.466,0.530,0.737,38.798,552.0
27,1h,Vanilla LSTM,all,all,161.942,162.454,162.482,166.186,168.569,139.185,-0.230,0.058,622.022,2184.0


Dataset: 1h_top4


,Experiment,Model,Feature set,Period,60m RMSE (mm),180m RMSE (mm),360m RMSE (mm),720m RMSE (mm),1440m RMSE (mm),Mean MAE (mm),Mean NSE,Mean KGE,Mean Peak Error (mm),Mean Peak Timing (phút)
45,1h_top4,Adapter LSTM,water_level+flood_season,all,51.439,92.338,121.284,118.750,109.765,71.342,0.536,0.521,484.969,684.0
48,1h_top4,Adapter LSTM,water_level+rain+flood_season,all,111.634,132.368,143.447,139.536,136.439,97.031,0.207,0.341,537.933,5016.0
51,1h_top4,LightGBM,water_level+flood_season,all,29.915,83.558,119.306,120.927,137.899,68.930,0.450,0.697,233.751,588.0
54,1h_top4,LightGBM,water_level+rain,all,35.956,91.321,139.279,170.783,242.653,79.274,-0.150,0.538,722.116,588.0
57,1h_top4,LightGBM,water_level+rain+flood_season,all,35.943,93.749,141.215,172.051,247.847,80.750,-0.189,0.525,731.319,612.0
60,1h_top4,Residual LSTM,water_level,all,20.621,52.220,78.432,88.065,105.122,48.058,0.736,0.799,112.307,564.0
63,1h_top4,Residual LSTM,water_level+flood_season,all,21.021,52.961,77.553,85.922,104.729,48.555,0.741,0.773,109.879,564.0
66,1h_top4,Residual LSTM,water_level+rain,all,22.745,59.932,85.713,81.414,100.671,46.264,0.744,0.830,59.927,576.0
69,1h_top4,Residual LSTM,water_level+rain+flood_season,all,21.073,52.203,76.757,78.184,96.894,45.093,0.773,0.834,74.150,480.0
72,1h_top4,Vanilla LSTM,water_level+flood_season,all,37.923,57.742,83.176,92.443,108.246,53.735,0.710,0.663,296.738,588.0


=== Mùa lũ (tháng 9–12) — 1 đơn vị RMSE = 1 mm ===
Dataset: 1h


...

,Experiment,Model,Feature set,Period,60m RMSE (mm),180m RMSE (mm),360m RMSE (mm),720m RMSE (mm),1440m RMSE (mm),Mean MAE (mm),Mean NSE,Mean KGE,Mean Peak Error (mm),Mean Peak Timing (phút)
0,1h,Adapter LSTM,all,all,135.737,148.850,155.909,154.589,154.526,124.478,-0.026,0.220,573.551,1176.0
1,1h,Adapter LSTM,all,flood_season,151.082,160.483,168.188,169.468,168.634,134.682,0.072,0.192,573.563,1176.0
2,1h,Adapter LSTM,all,non_flood_season,94.990,120.266,125.691,116.601,118.904,102.665,-3.567,-0.125,67.525,6384.0
3,1h,Adapter LSTM,water_level,all,41.392,57.935,87.651,105.649,113.683,57.879,0.644,0.603,313.066,468.0
4,1h,Adapter LSTM,water_level,flood_season,49.007,67.293,102.771,122.278,129.212,66.967,0.661,0.605,313.066,468.0
5,1h,Adapter LSTM,water_level,non_flood_season,24.128,37.829,54.300,70.226,81.989,42.946,-0.062,0.415,19.484,2904.0
6,1h,Adapter LSTM,water_level+rain,all,121.257,130.274,141.157,147.565,144.683,100.684,0.142,0.237,625.164,2688.0
7,1h,Adapter LSTM,water_level+rain,flood_season,144.683,152.558,161.118,168.361,167.350,123.331,0.125,0.140,625.174,2688.0
8,1h,Adapter LSTM,water_level+rain,non_flood_season,37.256,59.163,83.805,87.921,76.265,52.272,-0.715,0.244,23.470,6576.0
9,1h,LightGBM,all,all,42.347,83.424,128.125,223.407,139.612,85.221,0.058,0.533,413.108,672.0


Chênh lệch RMSE của cùng model + feature set: số âm nghĩa là top4 có RMSE thấp hơn.


experiment_name,model_type,feature_set,period,horizon_min,1h,1h_top4,top4_minus_1h_mm
120,LightGBM,water_level+rain,all,60,44.771,35.956,-8.815
121,LightGBM,water_level+rain,all,180,92.188,91.321,-0.867
122,LightGBM,water_level+rain,all,360,137.530,139.279,1.749
123,LightGBM,water_level+rain,all,720,154.724,170.783,16.059
124,LightGBM,water_level+rain,all,1440,136.551,242.653,106.102
125,LightGBM,water_level+rain,flood_season,60,56.229,43.582,-12.647
126,LightGBM,water_level+rain,flood_season,180,109.167,108.458,-0.709
127,LightGBM,water_level+rain,flood_season,360,153.614,161.294,7.680
128,LightGBM,water_level+rain,flood_season,720,179.129,205.262,26.133
129,LightGBM,water_level+rain,flood_season,1440,157.274,305.245,147.971


Best feature set của từng model trong mỗi dataset (không hard-code kết quả):


,Experiment,Model,Feature set,Period,60m RMSE (mm),180m RMSE (mm),360m RMSE (mm),720m RMSE (mm),1440m RMSE (mm),Mean MAE (mm),Mean NSE,Mean KGE,Mean Peak Error (mm),Mean Peak Timing (phút)
0,1h,Adapter LSTM,water_level,all,41.392,57.935,87.651,105.649,113.683,57.879,0.644,0.603,313.066,468.0
1,1h,Adapter LSTM,water_level,flood_season,49.007,67.293,102.771,122.278,129.212,66.967,0.661,0.605,313.066,468.0
2,1h,Adapter LSTM,water_level,non_flood_season,24.128,37.829,54.300,70.226,81.989,42.946,-0.062,0.415,19.484,2904.0
3,1h,LightGBM,water_level,all,34.116,84.687,117.284,122.001,136.657,68.583,0.445,0.681,227.663,600.0
4,1h,LightGBM,water_level,flood_season,41.644,100.735,133.805,138.870,166.822,79.465,0.470,0.691,227.663,600.0
5,1h,LightGBM,water_level,non_flood_season,17.924,52.688,87.335,91.562,71.773,52.506,-0.382,0.383,9.671,516.0
6,1h,Residual LSTM,water_level,all,19.350,50.589,75.825,82.019,98.438,45.493,0.762,0.803,107.533,588.0
7,1h,Residual LSTM,water_level,flood_season,22.924,60.561,89.822,94.216,115.751,54.903,0.762,0.806,107.521,588.0
8,1h,Residual LSTM,water_level,non_flood_season,10.524,25.011,41.283,54.567,56.750,29.077,0.444,0.616,12.697,324.0
9,1h,Vanilla LSTM,water_level,all,38.222,57.334,84.201,100.820,115.973,55.219,0.680,0.711,259.664,684.0


In [10]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
exports = {
    "mlflow_metrics_summary.csv": summary,
    "mlflow_metrics_by_horizon.csv": by_horizon,
    "mlflow_best_runs.csv": best_runs.drop(columns=["start_time"]),
    "mlflow_metrics_by_period.csv": by_period,
    "mlflow_period_comparison.csv": period_comparison,
    "mlflow_rmse_experiment_comparison.csv": rmse_comparison,
    "mlflow_evaluation_audit.csv": evaluation_audit,
}
export_paths = []
for filename, frame in exports.items():
    output = OUTPUT_DIR / filename
    frame.to_csv(output, index=False, encoding="utf-8-sig", na_rep="NaN")
    export_paths.append(output)
    print("Đã xuất:", output)

excel_engine = next((engine for engine in ["openpyxl", "xlsxwriter"]
                     if importlib.util.find_spec(engine) is not None), None)
if excel_engine:
    workbook = OUTPUT_DIR / "mlflow_metrics_report.xlsx"
    with pd.ExcelWriter(workbook, engine=excel_engine) as writer:
        best_runs.drop(columns=["start_time"]).to_excel(writer, sheet_name="Best Runs", index=False)
        summary_report.to_excel(writer, sheet_name="Summary", index=False)
        by_horizon.to_excel(writer, sheet_name="By Horizon", index=False)
        by_period.to_excel(writer, sheet_name="By Period", index=False)
        period_comparison.to_excel(writer, sheet_name="Period Comparison", index=False)
        evaluation_audit.to_excel(writer, sheet_name="Evaluation Audit", index=False)
        for period, table in period_tables.items():
            table.to_excel(writer, sheet_name={"all": "All Test", "flood_season": "Flood Season", "non_flood_season": "Non Flood Season"}[period], index=False)
        for metric, table in pivot_tables.items():
            table.to_excel(writer, sheet_name=PIVOT_LABELS[metric], index=False)
    export_paths.append(workbook)
    print("Đã xuất:", workbook)
else:
    print("Bỏ qua Excel vì môi trường chưa có openpyxl/xlsxwriter; các file CSV đã được xuất đầy đủ.")

Đã xuất: /home/my_ubuntu/projects/hydrology/scripts/mlflow_metrics_summary.csv
Đã xuất: /home/my_ubuntu/projects/hydrology/scripts/mlflow_metrics_by_horizon.csv
Đã xuất: /home/my_ubuntu/projects/hydrology/scripts/mlflow_best_runs.csv
Đã xuất: /home/my_ubuntu/projects/hydrology/scripts/mlflow_metrics_by_period.csv
Đã xuất: /home/my_ubuntu/projects/hydrology/scripts/mlflow_period_comparison.csv
Đã xuất: /home/my_ubuntu/projects/hydrology/scripts/mlflow_rmse_experiment_comparison.csv
Đã xuất: /home/my_ubuntu/projects/hydrology/scripts/mlflow_evaluation_audit.csv
Bỏ qua Excel vì môi trường chưa có openpyxl/xlsxwriter; các file CSV đã được xuất đầy đủ.


In [11]:
print("=== TỔNG KẾT ===")
print("Số experiment đã tìm thấy:", len(all_experiments))
print("Số experiment đã đọc run:", len(experiments))
print("Số run FINISHED:", len(finished_runs))
print("Số run sau khi filter model type:", len(model_runs_df))
print("Số run không đủ metadata/test_loss:", len(excluded_runs))
print("Số best run cuối cùng:", len(best_runs))
print("Danh sách model type:", sorted(model_runs_df["model_type"].dropna().unique()))
print("Danh sách feature set:", sorted(model_runs_df["feature_set"].dropna().unique()))
print("Bảng best run:")
display(best_runs.drop(columns=["start_time"]).round({"test_loss": 6}))
print("Bảng summary:")
display(summary_report.round(3))
print("Best feature set của mỗi model:")
display(best_feature_sets[["model_type", "feature_set", "test_loss", "experiment_name", "run_id"]].round({"test_loss": 6}))

print("1 đơn vị RMSE = 1 mm; MAE/Peak Error: mm; NSE/KGE: không đơn vị.")
print("Experiment dùng trong báo cáo:", EXPERIMENT_NAMES)
print("Run xác minh được inference:", int(evaluation_audit.replay_verified.sum()), "/", len(best_runs))
print("Metric theo mùa thiếu:", int(by_period.loc[by_period.period.ne("all"), METRICS].isna().sum().sum()))
for period in PERIODS:
    print(PERIOD_LABELS[period])
    display(period_tables[period].round(3))

=== TỔNG KẾT ===
Số experiment đã tìm thấy: 5
Số experiment đã đọc run: 2
Số run FINISHED: 116
Số run sau khi filter model type: 116
Số run không đủ metadata/test_loss: 0
Số best run cuối cùng: 30
Danh sách model type: ['Adapter LSTM', 'LightGBM', 'Residual LSTM', 'Vanilla LSTM', 'XGBoost']
Danh sách feature set: ['all', 'water_level', 'water_level+flood_season', 'water_level+rain', 'water_level+rain+flood_season']
Bảng best run:


,experiment_name,run_id,run_name,model_type,feature_set,test_loss,test_loss_scale,dataset_dir,resolution,lifecycle_stage,model_source,feature_source
0,1h,3d0cf00162384df68ae69b81faf99341,lstm_adapter_1h_all_two_stage,Adapter LSTM,all,0.416685,scaled_target_units,unknown,1h,active,params.train_method,params.feature_set
1,1h,ee746b6d99f24e7eab1c4c5b7f8458fc,lstm_adapter_1h_water_level_unfrozen,Adapter LSTM,water_level,0.136669,scaled_target_units,unknown,1h,deleted,params.train_method,params.feature_set
2,1h,4909104e216149788b094e3acfc4a439,lstm_adapter_1h_water_level+rain_two_stage,Adapter LSTM,water_level+rain,0.348901,scaled_target_units,unknown,1h,active,params.train_method,params.feature_set
3,1h,3b3b88e73c9f4e4289d6a4bd04f64fce,lgbm_1h_all,LightGBM,all,18914.299682,original_target_units,unknown,1h,active,params.train_method,params.feature_set
4,1h,73470359f6ec414abe8fb6a6988aa0d4,lgbm_1h_water_level,LightGBM,water_level,11130.172669,original_target_units,unknown,1h,active,params.train_method,params.feature_set
5,1h,86e2c54a3d5b41f09ec85f2791899ae5,lgbm_1h_water_level+rain,LightGBM,water_level+rain,14400.617792,original_target_units,unknown,1h,active,params.train_method,params.feature_set
6,1h,f4a41c88a1374c97bd35599a899d813e,residual_lstm_1h_all,Residual LSTM,all,0.180603,scaled_target_units,unknown,1h,active,params.train_method,params.feature_set
7,1h,4335acb3a3704626a782ea3635ce4204,residual_lstm_1h_water_level,Residual LSTM,water_level,0.093164,scaled_target_units,unknown,1h,active,params.train_method,params.feature_set
8,1h,306db755bc50422c8ff63af4258ee759,residual_lstm_1h_water_level+rain,Residual LSTM,water_level+rain,0.182173,scaled_target_units,unknown,1h,active,params.train_method,params.feature_set
9,1h,1be43fe5bedd4e1aab79e28d4228414d,lstm_1h_all,Vanilla LSTM,all,0.499385,scaled_target_units,unknown,1h,active,params.train_method,params.feature_set


Bảng summary:


,Experiment,Model,Feature set,Test loss,Mean RMSE,Mean MAE,Mean NSE,Mean KGE,Mean Peak Absolute Error,Mean Peak Timing Error
0,1h,Adapter LSTM,water_level,0.137,81.262,57.879,0.644,0.603,313.066,468.0
1,1h,Adapter LSTM,water_level+rain,0.349,136.987,100.684,0.142,0.237,625.164,2688.0
2,1h,Adapter LSTM,all,0.417,149.922,124.478,-0.026,0.220,573.551,1176.0
3,1h,LightGBM,water_level,11130.173,98.949,68.583,0.445,0.681,227.663,600.0
4,1h,LightGBM,water_level+rain,14400.618,113.153,84.441,0.282,0.619,235.212,648.0
5,1h,LightGBM,all,18914.300,123.383,85.221,0.058,0.533,413.108,672.0
6,1h,Residual LSTM,water_level,0.093,65.244,45.493,0.762,0.803,107.533,588.0
7,1h,Residual LSTM,all,0.181,93.050,65.352,0.533,0.755,21.369,552.0
8,1h,Residual LSTM,water_level+rain,0.182,93.437,65.466,0.530,0.737,38.798,552.0
9,1h,Vanilla LSTM,water_level,0.132,79.310,55.219,0.680,0.711,259.664,684.0


Best feature set của mỗi model:


...

,Experiment,Model,Feature set,Period,60m RMSE (mm),180m RMSE (mm),360m RMSE (mm),720m RMSE (mm),1440m RMSE (mm),Mean MAE (mm),Mean NSE,Mean KGE,Mean Peak Error (mm),Mean Peak Timing (phút)
0,1h,Adapter LSTM,all,all,135.737,148.850,155.909,154.589,154.526,124.478,-0.026,0.220,573.551,1176.0
3,1h,Adapter LSTM,water_level,all,41.392,57.935,87.651,105.649,113.683,57.879,0.644,0.603,313.066,468.0
6,1h,Adapter LSTM,water_level+rain,all,121.257,130.274,141.157,147.565,144.683,100.684,0.142,0.237,625.164,2688.0
9,1h,LightGBM,all,all,42.347,83.424,128.125,223.407,139.612,85.221,0.058,0.533,413.108,672.0
12,1h,LightGBM,water_level,all,34.116,84.687,117.284,122.001,136.657,68.583,0.445,0.681,227.663,600.0
15,1h,LightGBM,water_level+rain,all,44.771,92.188,137.530,154.724,136.551,84.441,0.282,0.619,235.212,648.0
18,1h,Residual LSTM,all,all,34.491,92.883,137.622,100.453,99.800,65.352,0.533,0.755,21.369,552.0
21,1h,Residual LSTM,water_level,all,19.350,50.589,75.825,82.019,98.438,45.493,0.762,0.803,107.533,588.0
24,1h,Residual LSTM,water_level+rain,all,34.384,92.569,137.794,101.200,101.235,65.466,0.530,0.737,38.798,552.0
27,1h,Vanilla LSTM,all,all,161.942,162.454,162.482,166.186,168.569,139.185,-0.230,0.058,622.022,2184.0


Mùa lũ (tháng 9–12)


,Experiment,Model,Feature set,Period,60m RMSE (mm),180m RMSE (mm),360m RMSE (mm),720m RMSE (mm),1440m RMSE (mm),Mean MAE (mm),Mean NSE,Mean KGE,Mean Peak Error (mm),Mean Peak Timing (phút)
1,1h,Adapter LSTM,all,flood_season,151.082,160.483,168.188,169.468,168.634,134.682,0.072,0.192,573.563,1176.0
4,1h,Adapter LSTM,water_level,flood_season,49.007,67.293,102.771,122.278,129.212,66.967,0.661,0.605,313.066,468.0
7,1h,Adapter LSTM,water_level+rain,flood_season,144.683,152.558,161.118,168.361,167.350,123.331,0.125,0.140,625.174,2688.0
10,1h,LightGBM,all,flood_season,53.300,101.051,148.062,276.360,155.740,100.564,0.066,0.521,413.108,672.0
13,1h,LightGBM,water_level,flood_season,41.644,100.735,133.805,138.870,166.822,79.465,0.470,0.691,227.663,600.0
16,1h,LightGBM,water_level+rain,flood_season,56.229,109.167,153.614,179.129,157.274,96.212,0.341,0.620,235.212,648.0
19,1h,Residual LSTM,all,flood_season,41.865,113.049,165.930,109.478,119.328,79.850,0.527,0.753,21.372,552.0
22,1h,Residual LSTM,water_level,flood_season,22.924,60.561,89.822,94.216,115.751,54.903,0.762,0.806,107.521,588.0
25,1h,Residual LSTM,water_level+rain,flood_season,41.626,112.631,166.270,110.443,120.088,79.565,0.524,0.735,38.805,552.0
28,1h,Vanilla LSTM,all,flood_season,172.043,172.117,171.338,175.014,176.201,141.876,-0.040,0.051,621.965,2184.0


Ngoài mùa lũ


,Experiment,Model,Feature set,Period,60m RMSE (mm),180m RMSE (mm),360m RMSE (mm),720m RMSE (mm),1440m RMSE (mm),Mean MAE (mm),Mean NSE,Mean KGE,Mean Peak Error (mm),Mean Peak Timing (phút)
2,1h,Adapter LSTM,all,non_flood_season,94.990,120.266,125.691,116.601,118.904,102.665,-3.567,-0.125,67.525,6384.0
5,1h,Adapter LSTM,water_level,non_flood_season,24.128,37.829,54.300,70.226,81.989,42.946,-0.062,0.415,19.484,2904.0
8,1h,Adapter LSTM,water_level+rain,non_flood_season,37.256,59.163,83.805,87.921,76.265,52.272,-0.715,0.244,23.470,6576.0
11,1h,LightGBM,all,non_flood_season,15.676,46.430,91.004,103.996,111.599,62.554,-0.917,0.302,13.284,576.0
14,1h,LightGBM,water_level,non_flood_season,17.924,52.688,87.335,91.562,71.773,52.506,-0.382,0.383,9.671,516.0
17,1h,LightGBM,water_level+rain,non_flood_season,17.169,58.721,109.527,109.099,98.238,67.052,-1.121,0.280,20.084,816.0
20,1h,Residual LSTM,all,non_flood_season,14.397,37.158,62.821,82.372,49.937,40.055,0.036,0.509,1.976,552.0
23,1h,Residual LSTM,water_level,non_flood_season,10.524,25.011,41.283,54.567,56.750,29.077,0.444,0.616,12.697,324.0
26,1h,Residual LSTM,water_level+rain,non_flood_season,14.890,37.226,62.282,82.633,54.477,40.865,0.006,0.501,7.079,552.0
29,1h,Vanilla LSTM,all,non_flood_season,137.900,139.581,141.712,145.537,150.971,133.439,-5.981,-0.006,62.237,5928.0
